<a href="https://colab.research.google.com/github/THEJoshinator20/ST-554-Project1-Template/blob/main/Task1/Task1_notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Alana Pooler
<br>
ST 554 Project 1

# Task 1: Prediction of C6H6(GT)

This task involves writing two gradient descent type algorithms to find the optimal constant to use for squared error loss (ends up being the sample mean) and to find the optimal intercept and slope from a simple linear regression model.

## Load and clean data set

In [ ]:
# install ucimlrepo
!pip install ucimlrepo

In [138]:
# import libraries
import ucimlrepo as uci
import numpy as np
import pandas as pd
from typing import Optional
from sklearn import linear_model

In [ ]:
air_quality = uci.fetch_ucirepo(id=360)
# view data set
air_quality = air_quality.data.features
air_quality.head()

,Date,Time,CO(GT),PT08.S1(CO),NMHC(GT),C6H6(GT),PT08.S2(NMHC),NOx(GT),PT08.S3(NOx),NO2(GT),PT08.S4(NO2),PT08.S5(O3),T,RH,AH
0,3/10/2004,18:00:00,2.6,1360,150,11.9,1046,166,1056,113,1692,1268,13.6,48.9,0.7578
1,3/10/2004,19:00:00,2.0,1292,112,9.4,955,103,1174,92,1559,972,13.3,47.7,0.7255
2,3/10/2004,20:00:00,2.2,1402,88,9.0,939,131,1140,114,1555,1074,11.9,54.0,0.7502
3,3/10/2004,21:00:00,2.2,1376,80,9.2,948,172,1092,122,1584,1203,11.0,60.0,0.7867
4,3/10/2004,22:00:00,1.6,1272,51,6.5,836,131,1205,116,1490,1110,11.2,59.6,0.7888


Remove any observations where the C6H6(GT) or CO(GT) are -200 as these represent missing values

In [ ]:
air_quality_clean = air_quality[(air_quality['C6H6(GT)'] != -200) & (air_quality['CO(GT)'] != -200)]

Since we will mainly be using C6H6(GT) as y and PT08.S1(CO) as x, rename these variables to be easier to reference

In [ ]:
air_quality_clean = air_quality_clean.rename(columns = {'C6H6(GT)': 'y', 'PT08.S1(CO)': 'x'})

## Grid Search Algorithm: Just y

Implement a grid search to find the optimal value of c based off the data set.

 Create function to calculate RMSE

 We will generalize this function so that it can be used in all four parts of this task: Grid search algorithm with just y, Grid search algorithm with x and y, gradient descent algorithm with just y, and gradient descent algorithm with x and y.

In [ ]:
def rmse(
    y: pd.Series,
    x: Optional[pd.Series] = None,
    c: Optional[float] = None,
    b0: Optional[float] = None,
    b1: Optional[float] = None
) -> float:
  """
  Calculates Root Mean Squared Error (RMSE) using either just a y variable or x and y variables.
  """
  # check if x, b0, and b1 have been given - calculate RMSE using these values if so
  if x is not None and b0 is not None and b1 is not None:
    return np.sqrt(np.mean((y - b0 - b1 * x) ** 2))

  # otherwise, calculate RMSE using just y and c
  else:
    return np.sqrt(np.mean((y - c) ** 2))

Write function to find optimal c given y variable:


*   Find the first and third quartiles of C6H6(GT) to determine reasonable values for grid search
*   Use a list comprehensive to loop over the grid of c values, finding the RMSE for each value of c.
*   Determine which value of c gives the optimal (smallest) RMSE.
*   Report that as the prediction





In [181]:
def find_optimal_c(y: pd.Series) -> float:
  """
  Function to find the optimal value of c for a given column.
  """
  # find first and third quartile
  q1, q3 = np.percentile(y, [25, 75])

  # create grid using quartiles
  grid = np.linspace(q1, q3, 100)

  # calculate RMSE for each value in grid
  rmse_values = [rmse(y, c = c) for c in grid]

  # output optimal value of c
  return grid[np.argmin(rmse_values)]

Test function on C6H6(GT)

According to calculus, the optimal value of c should be the mean of y. The mean of C6H6(GT) is 10.2757, so the grid search algorithm did a pretty good job.

In [182]:
c = find_optimal_c(air_quality_clean['y'])
print(f"Optimal c value = {round(c, 4)}")
print(f"Mean value of C6H6(GT) = {round(air_quality_clean['y'].mean(), 4)}")

Optimal c value = 10.2828
Mean value of C6H6(GT) = 10.2757


Test function on PT08.S1(CO) to make sure algorithm generalizes

Again, the optimal value of c found by the grid search algorithm is very close to the mean value of PT08.S1(CO), so we can conclude that the algorithm generalizes well.

In [ ]:
c = find_optimal_c(air_quality_clean['x'])
print(f"Optimal c value = {round(c, 4)}")
print(f"Mean value of PT08.S1(CO) = {round(air_quality_clean['x'].mean(), 4)}")

Optimal c value = 1109.6364
Mean value of PT08.S1(CO) = 1110.5807


## Grid search algorithm: x and y

Implement the grid search to find the optimal pair of values for b0 and b1 using PT08.S1(CO) as your x variable and C6H6(GT) as your y variable.


Function to find optimal b0 and b1 values:

*   Populate a grid of b0 and b1 values to consider
*   Calulcate RMSE using x, y, b0, and b1
*   Report the optimal b0 and b1 combination based on RMSE

In [179]:
def find_optimal_betas(y: pd.Series, x: pd.Series):
  """
  Finds optimal values of b0 and b1 for a given x and y variable.
  """

  # create b0 and b1 values
  b0_vals, b1_vals = np.arange(-25, -15, 0.1), np.arange(-5, 5, 0.01)

  # create grid of b0 and b1 values
  grid = [(b0, b1) for b0 in b0_vals for b1 in b1_vals]

  # calculate RMSE for each value of b0 and b1
  rmse_values = [rmse(y, x, b0 = b0, b1 = b1) for b0, b1 in grid]

  # find best values of b0 and b1
  best_b0, best_b1 = grid[np.argmin(rmse_values)]

  return best_b0, best_b1


Test algorithm using PT08.S1(CO) as x and C6H6(GT) as y:

In [180]:
b0, b1 = find_optimal_betas(air_quality_clean['y'], air_quality_clean['x'])
print(f"Optimal b0 = {round(b0, 4)}, optimal b1 = {round(b1, 4)}")

Optimal b0 = -23.0, optimal b1 = 0.03


Use the values found for b0 and b1 to predict a new C6H6(GT) for a PT08.S1(CO) of 946, 1075, and 1246.

Predicted value for PT08.S1(CO) = 946: 5.38
<br>
Predicted value for PT08.S1(CO) = 1075: 9.25
<br>
Predicted value for PT08.S1(CO) = 1246: 14.38

In [ ]:
# use for loop to calculate values of C6H6(GT) using given values of PT08.S1(CO)
for num in [946, 1075, 1246]:
  pred = b0 + b1 * num
  print(round(pred, 4))

5.38
9.25
14.38


## Gradient Descent Algorithm: Just y

Function to calculate difference quotient to approximate the slope of the tangent line

We will generalize this function so that it can be used in the gradient descent algorithm with just y, as well as the gradient descent algorithm with x and y.

In [174]:
def diff_quotient(
    y: pd.Series,
    delta: float,
    param: str,
    x: Optional[pd.Series] = None,
    c: Optional[float] = None,
    b0: Optional[float] = None,
    b1: Optional[float] = None
) -> float:
  """
  Calculate difference quotient to approximate the slope of tangent line using a given y, c, and delta, or a given x, y, b0, b1, and delta.
  """

  # calculte RMSE without delta
  base_rmse = rmse(y = y, x = x, c = c, b0 = b0, b1 = b1)

  if param == 'c':
    rmse_delta = rmse(y = y, c = c + delta)

  elif param == 'b0':
    rmse_delta = rmse(y = y, x = x, b0 = b0 + delta, b1 = b1)

  elif param == 'b1':
    rmse_delta = rmse(y = y, x = x, b0 = b0, b1 = b1 + delta)

  # return difference quotient
  return (rmse_delta - base_rmse) / delta

Gradient descent algorithm:



* Pick a starting value for c and assign it to cur_c
* Evaluate the difference quotient at cur_c
* Update the value by moving a small step in the negative  direction of the difference quotient
* Check if abs(new_c - cur_c) < num_tol where num_tol is a small value.
*  If so, update the cur_c to be the new_c value and stop.
* If not, update the value of cur_c to new_c and repeat steps 5-7.
* Put in a safety that stops the loop after a maximum number of iterations is reached (even if
abs(new_c - cur_c) < num_tol is not met)
* Use the last value as the prediction!



In [175]:
def gradient_descent_y(
    y: pd.Series,
    start_value: int = 0,
    delta: float = 0.001,
    step_size: float = 0.01,
    tolerance: float = 0.0001,
    max_iterations: int = 10000
) -> float:
  """
  Calculate optimal value of c using a gradient descent algorithm.
  """

  # assign starting value to cur_c
  cur_c = start_value

  for i in range(max_iterations):
    # calculate new_c
    new_c = cur_c - (diff_quotient(y, delta, param = 'c', c = cur_c)) * step_size

    # stop loop if abs(new_c - cur_c) < num_tol
    if abs(new_c - cur_c) < tolerance:
      break

    # otherwise, update cur_c and repeat
    cur_c = new_c

  return cur_c

Test function on C6H6(GT), using 0 as starting value

The result is very similar to the value of c we got from the grid search algorithm, although the grid search algorithm resulted in a c value that was closer to the mean of C6H6(GT).

In [176]:
c = gradient_descent_y(air_quality_clean['y'], start_value = 0)
print(f"Optimal c value for C6H6(GT) = {round(c, 4)}")

Optimal c value for C6H6(GT) = 10.2009


Test function on PT08.S1(CO) to make sure algorithm generalizes, using 1100 as the start value

This value of c is closer to the mean of PT08.S1(CO) than the c value we got from the grid search algorithm. We can conclude that the algorithm does generalize well.

In [166]:
c = gradient_descent_y(air_quality_clean['x'], start_value = 1100, step_size = 0.1)
print(f"Optimal c value for PT08.S1(CO) = {round(c, 4)}")

Optimal c value for PT08.S1(CO) = 1110.3616


## Gradient Descent Algorithm: Using x and y

Implement a gradient descent algorithm to find the optimal b0 and b1 values using C6H6(GT) as y and PT08.S1(CO) as x.

Function to run the gradient descent algorithm using x and y:

*   Pick starting values for b_0 and b_1 (say cur_b0 and cur_b1)
*   Evaluate the difference quotient for b_0 at the cur_b0 and cur_b1 values
*   Update the value of b_0 by moving (a small step) in the negative direction of the difference quotient
new_b0 = cur_b0 - diff_quotient_b0 * step_size_b0
*   Evaluate the difference quotient for b_1 at the new_b0 and cur_b1 values
*   Update the value of b_1 by moving (a small step) in the negative direction of the difference quotient
new_b1 = cur_b1 - diff_quotient_b1 * step_size_b1
*   Check if the distance between the cur_b0, cur_b1 vector to the new_b0, new_b1 vector is less than
some small tolerance
*   If so, update the cur_b0 and cur_b1 to be the new_b0 and new_b1 values and stop
*   If not, update the cur_b0 and cur_b1 to be the new_b0 and new_b1 values and repeat steps 6-10
*   Put in a safety that stops the loop after a maximum number of iterations is reached (even if
the tolerance is not met)
*   Use the last values as the estimates for b_0 and b_1
*   Use initial starting values of -20 for the intercept and 0 for the slope

In [187]:
def gradient_descent_xy(
    y: pd.Series,
    x: pd.Series,
    delta_b0: float = 0.005,
    delta_b1: float = 0.005,
    start_b0: int = -20,
    start_b1: int = 0,
    step_size_b0: float =  0.5,
    step_size_b1: float = 0.00005,
    tolerance: float = 0.0001,
    max_iterations: int = 100000
):
  """
  Run a gradient descent algorithm to find optimal b0 and b1 values.
  """

  # get starting values for b0 and b1
  cur_b0, cur_b1 = start_b0, start_b1

  # loop to keep calculating b0 and b1 until max # of iterations is reached
  for i in range(max_iterations):

    # calculate difference quotient for b0
    diff_quotient_b0 = diff_quotient(y, delta_b0, 'b0', x, b0 = cur_b0, b1 = cur_b1)
    # calculate new b0 value
    new_b0 = cur_b0 - diff_quotient_b0 * step_size_b0

    # calculate difference quotient for b1 using new b0 value
    diff_quotient_b1 = diff_quotient(y, delta_b1, 'b1', x, b0 = new_b0, b1 = cur_b1)
    # calculate new b1
    new_b1 = cur_b1 - diff_quotient_b1 * step_size_b1

    # calculate euclidean distance between current b0, b1 and new b0, b1
    distance = np.linalg.norm(np.array([new_b0, new_b1]) - np.array([cur_b0, cur_b1]))

    # stop loop if euclidian distance is less than tolerance
    if distance < tolerance:
      break

    # otherwise, update cur_c and repeat
    cur_b0, cur_b1 = new_b0, new_b1

  return cur_b0, cur_b1

In [ ]:
gd_b0, gd_b1 = gradient_descent_xy(y = air_quality_clean['y'], x = air_quality_clean['x'])
gd_b0, gd_b1

Use values of b0 and b1 found by gradient descent algorithm to predict a new C6H6(GT) for a PT08.S1(CO) of 946, 1075, and 1246.

In [ ]:
# use for loop to calculate new values of C6H6(GT) using given values of PT08.S1(CO)
for num in [946, 1075, 1246]:
  pred = gd_b0 + gd_b1 * num
  print(round(pred, 4))

Find values of b0 and b1 using calculus based method to compare to results of gradient descent algorithm

The values of b0 and b1 that we got from the gradient descent algorithm are very close to the values given by the SLR model, and the values given by the grid search algorithm are closer (-23 and 0.03)

In [143]:
reg = linear_model.LinearRegression()
reg.fit(air_quality_clean['x'].values.reshape(-1,1), air_quality_clean['y'])
print(f"Optimal b0 from SLR = {reg.intercept_}")
print(f"Optimal b1 from SLR = {reg.coef_}")

Optimal b0 from SLR = -22.95574738007751
Optimal b1 from SLR = [0.02992262]


Takeaways:

The grid search algorithm performed slightly better than the gradient descent algorithm when using just y as well as when using x and y. The values found for c, b0 and b1 were slightly closer to the values found using calculus based methods than the gradient descent algorithm was.

The run time for the grid search algorithm is also shorter, especially when using x and y, compared to the run time for the gradient descent algorithm.